In [ ]:
!pip install accelerate bitsandbytes==0.46.1 transformers datasets peft trl==0.19.1

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import (
    LoraConfig,
    PeftModel,
    TaskType,
    get_peft_model,
)
from trl import DataCollatorForCompletionOnlyLM, SFTConfig, SFTTrainer
from torch.utils.data import DataLoader
import torch

import numpy as np
import random

SEED = 42

def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


seed_everything(42)
PAD_TOKEN = "<|pad|>"
seq_length = 1800

OUTPUT_DIR ="model_training"

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
)
# Define your saved path
model_path = "meta-llama/Llama-3.2-1B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_path)

tokenizer.add_special_tokens({"pad_token": PAD_TOKEN})
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(model_path,quantization_config=quant_config)


model.resize_token_embeddings(len(tokenizer), pad_to_multiple_of=8)

In [ ]:
# model.save_pretrained(r"E:\MCQ_generation\Offline Model\Llama 3.2 1B")
# tokenizer.save_pretrained(r"E:\MCQ_generation\Offline Model\Llama 3.2 1B")

In [ ]:
# import pandas as pd


# train_pd = pd.read_json(r"E:\MCQ_generation\Dataset\Final Format\train.json")
# dev_pd = pd.read_json(r"E:\MCQ_generation\Dataset\Final Format\dev.json")
# test_pd = pd.read_json(r"E:\MCQ_generation\Dataset\Final Format\test.json")



In [ ]:
# def format_example(row: dict):
#     messages = [
#         {
#             "role": "system",
#             "content": f"""Your Job is to generate Multiple choice questions from the provided context""",
#         },
#         {"role": "user", "content": row["query"]},
#         {"role": "assistant", "content": row["response"]},
#     ]
#     return tokenizer.apply_chat_template(messages, tokenize=False)

In [ ]:
# train_pd["text"] = train_pd.apply(format_example, axis=1)
# dev_pd["text"] = dev_pd.apply(format_example, axis=1)
# test_pd["text"] = test_pd.apply(format_example, axis=1)

# train_pd.to_json(r"E:\MCQ_generation\Dataset\Final Format\train.json", orient="records", indent=2, force_ascii=False)
# test_pd.to_json(r"E:\MCQ_generation\Dataset\Final Format\test.json", orient="records", indent=2, force_ascii=False)
# dev_pd.to_json(r"E:\MCQ_generation\Dataset\Final Format\dev.json", orient="records", indent=2, force_ascii=False)


In [ ]:
# def count_tokens(row: dict) -> int:
#     return len(
#         tokenizer(
#             row["text"],
#             add_special_tokens=True,
#             return_attention_mask=False,
#         )["input_ids"]
#     )

In [ ]:
# train_pd["token_count"] = train_pd.apply(count_tokens, axis=1)
# test_pd["token_count"] = test_pd.apply(count_tokens, axis=1)
# dev_pd["token_count"] = dev_pd.apply(count_tokens, axis=1)

In [ ]:
# plt.hist(dev_pd.token_count, weights=np.ones(len(dev_pd.token_count)) / len(dev_pd.token_count))
# plt.gca().yaxis.set_major_formatter(PercentFormatter(1))
# plt.xlabel("Tokens")
# plt.ylabel("Percentage")
# plt.show()

In [ ]:
# seq_length = max(train_pd.token_count)
# max(train_pd.token_count)

In [ ]:
# Configure LoRA
lora_config = LoraConfig(
    r=32,                            # rank
    lora_alpha=128,
    target_modules=[
        "self_attn.q_proj",
        "self_attn.k_proj",
        "self_attn.v_proj",
        "self_attn.o_proj",
        "mlp.gate_proj",
        "mlp.up_proj",
        "mlp.down_proj",],      # T5 uses "q", "v" in attention
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

In [ ]:
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files={
    "train": "/content/train.json",
    "validation": "/content/dev.json",
    "test": "/content/test.json"
})
dataset

In [ ]:
# from tqdm import tqdm

# input_lengths = []
# output_lengths = []

# for example in tqdm(dataset["train"]):
#     input_ids = tokenizer(example["query"], truncation=False)["input_ids"]
#     label_ids = tokenizer(example["response"], truncation=False)["input_ids"]

#     input_lengths.append(len(input_ids))
#     output_lengths.append(len(label_ids))

# print("📏 Max input tokens:", max(input_lengths))
# print("📏 Max output tokens:", max(output_lengths))
# print("📊 95th percentile input:", sorted(input_lengths)[int(0.95 * len(input_lengths))])
# print("📊 95th percentile output:", sorted(output_lengths)[int(0.95 * len(output_lengths))])


In [ ]:
# for split in dataset:
#     print(f"🔹 {split} set: {len(dataset[split])} samples")

In [ ]:
# def filter_fn(example):
#     input_len = len(tokenizer(example["query"])["input_ids"])
#     output_len = len(tokenizer(example["response"])["input_ids"])
#     return input_len <= 1024 and output_len <= 512

# dataset["train"] = dataset["train"].filter(filter_fn)
# dataset["validation"] = dataset["validation"].filter(filter_fn)
# dataset["test"] = dataset["test"].filter(filter_fn)

In [ ]:
# for split in dataset:
#     print(f"🔹 {split} set: {len(dataset[split])} samples")

In [ ]:
response_template = "<|end_header_id|>"
collator = DataCollatorForCompletionOnlyLM(response_template, tokenizer=tokenizer)

examples = [dataset["train"][0]["text"]]
encodings = [tokenizer(e) for e in examples]

dataloader = DataLoader(encodings, collate_fn=collator, batch_size=1)

In [ ]:
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    dataset_text_field="text",
    max_seq_length=seq_length,
    num_train_epochs=5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=1,
    optim="paged_adamw_8bit",
    eval_strategy="steps",
    eval_steps=0.2,
    save_steps=0.2,
    logging_steps=10,
    learning_rate=1e-4,
    bf16=True,
    save_strategy="steps",
    warmup_ratio=0.1,
    save_total_limit=2,
    lr_scheduler_type="constant",
    report_to="none",
    save_safetensors=True,
    dataset_kwargs={
        "add_special_tokens": False,  # We template with special tokens
        "append_concat_token": False,  # No need to add additional separator token
    },
    seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    data_collator=collator,
)

In [ ]:
trainer.train()

In [ ]:
NEW_MODEL = "./llama-3.2-1B-mcq-gen-finetuned"

In [ ]:
# Save the LoRA adapter first
trainer.save_model(NEW_MODEL)

In [ ]:
# Load base model (FRESH, not quantized for merging)
base_model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="auto",
)

# Prepare tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_path)  # ✅ Load from base, not NEW_MODEL
tokenizer.add_special_tokens({"pad_token": PAD_TOKEN})
tokenizer.padding_side = "right"

# Resize embeddings to match training
base_model.resize_token_embeddings(len(tokenizer), pad_to_multiple_of=8)

In [ ]:
# Load LoRA adapter and merge
model_with_adapter = PeftModel.from_pretrained(base_model, NEW_MODEL)
merged_model = model_with_adapter.merge_and_unload()
print("Model merged and saved.")

In [ ]:
# Save merged model locally
MERGED_MODEL_DIR = "./llama-3.2-1B-mcq-gen-finetuned-merged"
merged_model.save_pretrained(MERGED_MODEL_DIR)
tokenizer.save_pretrained(MERGED_MODEL_DIR)
print("Model merged and saved.")

In [ ]:
## Create repo first

merged_model.push_to_hub(
    "sinister007/llama-3.2-1B-mcq-gen-finetuned",
    token=os.getenv("HF_TOKEN")
)
tokenizer.push_to_hub(
    "sinister007/llama-3.2-1B-mcq-gen-finetuned",
    token=os.getenv("HF_TOKEN")
)

print("Model and tokenizer uploaded to sinister007/llama-3.2-1B-mcq-gen-finetuned")